In [1]:
import time
import json
import os
import contextlib
import pandas as pd
from typing import TypedDict
from enum import Enum

import wave
import torch
import whisper
from batchalign.pipelines.pipeline import BatchalignPipeline
from batchalign import Document
from batchalign.formats.chat import CHATFile

In [2]:
class TorchBackend(Enum):
    MPS = "mps"  # Apple Silicon (M1/M2/M3/M4/M5 Macbooks)
    CUDA = "cuda"  # Nvidia (dedicated windows GPU)
    CPU = "cpu"  # CPU (fallback, slower)


class ModelDescription(TypedDict):
    name: str
    device: str
    backend: str
    dtype: str
    total_params: int
    ram_usage: str
    ram_usage_bytes: int


class WhisperModelSize(Enum):
    TINY = "tiny"
    BASE = "base"
    SMALL = "small"
    MEDIUM = "medium"
    LARGE = "large"
    TURBO = "turbo"


def get_device_and_dtype() -> tuple[TorchBackend, torch.dtype]:
    if torch.backends.mps.is_available():
        # Macbook with Apple Silicon
        device = TorchBackend.MPS
        torch_dtype = torch.float32
    elif torch.cuda.is_available():
        # Windows with Nvidia GPU
        device = TorchBackend.CUDA
        torch_dtype = torch.float16
    else:
        # CPU fallback
        device = TorchBackend.CPU
        torch_dtype = torch.float32
    return device, torch_dtype


def convert_model(nlp: BatchalignPipeline, torch_backend: TorchBackend) -> BatchalignPipeline:
    """Convertit le modèle Whisper du pipeline Batchalign pour qu'il utilise le backend spécifié."""\
    

    model = nlp.__dict__["_BatchalignPipeline__generator"].__dict__[
        "_OAIWhisperEngine__whisper"
    ]
    model_desc = get_model_description(
        model
    )

    if model_desc['backend'] == torch_backend.value:
        print(f"Model is already on the correct backend ({torch_backend.value}), no conversion needed.")
        return nlp
    else:
        print(f"Converting model from {model_desc['backend']} to {torch_backend.value}...")
    
    model = model.to(torch_backend.value)
    nlp.__dict__["_BatchalignPipeline__generator"].__dict__[
        "_OAIWhisperEngine__whisper"
    ] = model

    if torch_backend == TorchBackend.MPS:
        import whisper.timing as whisper_timing

        # Set Torch MPS fallback to allow using MPS even if some operations are not supported
        os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

        # Patch whisper DTW to use CPU because of unsupported operations on MPS
        _original_dtw = whisper_timing.dtw

        def dtw_mps_safe(x):
            if isinstance(x, torch.Tensor) and x.device.type == "mps":
                # print("Using CPU fallback for DTW on MPS device")
                return whisper_timing.dtw_cpu(x.detach().cpu().double().numpy())

            return _original_dtw(x)

        whisper_timing.dtw = dtw_mps_safe

    return nlp


def build_pipeline(lang="fra", num_speakers=2, model_size=WhisperModelSize.TURBO, torch_backend: TorchBackend = TorchBackend.CPU) -> tuple[BatchalignPipeline, ModelDescription, torch.dtype]:
    device, torch_dtype = get_device_and_dtype()
    nlp = BatchalignPipeline.new("asr", lang=lang, num_speakers=num_speakers)


    # Inject the model size
    model = whisper.load_model(model_size.value)
    nlp.__dict__["_BatchalignPipeline__generator"].__dict__[
        "_OAIWhisperEngine__whisper"
    ] = model

    # Convert the model to the correct device and dtype
    convert_model(nlp, torch_backend)

    # Get the model description
    model_desc = get_model_description(model)

    return nlp, model_desc, torch_dtype


def get_model_description(model: torch.nn.Module) -> ModelDescription:
    name = model.__class__.__name__
    first_param = next(model.parameters())
    device = first_param.device
    backend = first_param.device.type
    dtype = first_param.dtype
    total_params = sum(p.numel() for p in model.parameters())

    # Estimated RAM usage : total params x dtype
    ram_usage_bytes = total_params * torch.tensor([], dtype=dtype).element_size()

    def display_ram_usage(bytes: int) -> str:
        if bytes < 1024:
            return f"{bytes} B"
        elif bytes < 1024**2:
            return f"{bytes / 1024:.2f} KB"
        elif bytes < 1024**3:
            return f"{bytes / 1024**2:.2f} MB"
        else:
            return f"{bytes / 1024**3:.2f} GB"

    return {
        "name": name,
        "device": str(device),
        "backend": backend,
        "dtype": str(dtype),
        "total_params": total_params,
        "ram_usage": display_ram_usage(ram_usage_bytes),
        "ram_usage_bytes": ram_usage_bytes,
    }


def describe_model(nlp: BatchalignPipeline):
    model = nlp.__dict__["_BatchalignPipeline__generator"].__dict__[
        "_OAIWhisperEngine__whisper"
    ]
    model_desc = get_model_description(model)

    # Print it nicely
    print(f"### Model ###  " + "-" * 50)
    print(f'\t> Name  :\t{model_desc["name"]}')
    print(f"\t> Device:\t{model_desc['device']} ")
    print(f"\t> dtype :\t{model_desc['dtype']} ")
    print(f"\t> RAM   :\t{model_desc['ram_usage']}")


def get_wav_duration_seconds(wav_path: str) -> float:
    """Retourne la durée d'un fichier WAV en secondes."""
    with contextlib.closing(wave.open(wav_path, "r")) as f:
        frames = f.getnframes()
        rate = f.getframerate()
        if rate > 0:
            return frames / float(rate)
    return -1


nlp, device, torch_dtype = build_pipeline(model_size=WhisperModelSize.TURBO, torch_backend=TorchBackend.CPU)
describe_model(nlp)

Converting model from cuda to cpu...
### Model ###  --------------------------------------------------
	> Name  :	Whisper
	> Device:	cpu 
	> dtype :	torch.float32 
	> RAM   :	3.01 GB


In [3]:
class TranscriptionResult(TypedDict):
    audio_file: str
    torch_backend: str
    torch_dtype: str
    audio_duration_s: float
    pipeline_creation_time_s: float
    transcription_time_s: float


def transcribe_audio(
    audio_file: str, outfile: str, nlp: BatchalignPipeline | None = None, torch_backend: TorchBackend = TorchBackend.CPU, model_size: WhisperModelSize = WhisperModelSize.TURBO
) -> TranscriptionResult:

    # Build the pipeline
    if nlp is None:
        t0 = time.time()
        nlp, device, torch_dtype = build_pipeline(torch_backend=torch_backend, model_size=model_size)
        t_pipeline = time.time() - t0
    else:
        # Match model to backend
        nlp = convert_model(nlp, torch_backend)
        t_pipeline = 0.0
    desc = get_model_description(nlp.__dict__["_BatchalignPipeline__generator"].__dict__["_OAIWhisperEngine__whisper"])


    # Extract audio duration
    audio_duration = get_wav_duration_seconds(audio_file)

    # Create a Batchalign Document from the audio file
    doc = Document.new(media_path=audio_file, lang="fra")

    t0 = time.time()
    doc = nlp(doc)
    t_transcription = time.time() - t0

    chat = CHATFile(doc=doc)
    chat.write(outfile)

    print(f">>> Transcription of {audio_file} <<<")
    print(f"\t> Whisper backend:  \t{desc['backend']} ({desc['dtype']})")
    print(f"\t> Audio duration:   \t{audio_duration:.2f} s")
    print(f"\t> Pipeline creation:\t{t_pipeline:.2f} s")
    print(f"\t> Transcription time:\t{t_transcription:.2f} s")
    return TranscriptionResult(
        audio_file=audio_file,
        torch_backend=desc['backend'],
        torch_dtype=desc['dtype'],
        audio_duration_s=audio_duration,
        pipeline_creation_time_s=t_pipeline,
        transcription_time_s=t_transcription,
    )


transcribe_audio(
    "./data/input/sub-DC22212_ses-v1_task-wab-audio.wav",
    "./data/sample/output.cha",
    nlp,
    TorchBackend.CPU
)

Model is already on the correct backend (cpu), no conversion needed.


/home/phil/perso/transcription-audio/.venv/lib/python3.12/site-packages/whisper/transcribe.py:130: UserWarning: Performing inference on CPU when CUDA is available
  warnings.warn("Performing inference on CPU when CUDA is available")
/home/phil/perso/transcription-audio/.venv/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


>>> Transcription of ./data/input/sub-DC22212_ses-v1_task-wab-audio.wav <<<
	> Whisper backend:  	cpu (torch.float32)
	> Audio duration:   	52.98 s
	> Pipeline creation:	0.00 s
	> Transcription time:	26.68 s


{'audio_file': './data/input/sub-DC22212_ses-v1_task-wab-audio.wav',
 'torch_backend': 'cpu',
 'torch_dtype': 'torch.float32',
 'audio_duration_s': 52.984489795918364,
 'pipeline_creation_time_s': 0.0,
 'transcription_time_s': 26.684547901153564}

In [4]:
transcribe_audio(
    "./data/input/sub-DC22212_ses-v1_task-wab-audio.wav",
    "./data/sample/output.cha",
    nlp,
    TorchBackend.CUDA
)

Converting model from cpu to cuda...
>>> Transcription of ./data/input/sub-DC22212_ses-v1_task-wab-audio.wav <<<
	> Whisper backend:  	cuda (torch.float32)
	> Audio duration:   	52.98 s
	> Pipeline creation:	0.00 s
	> Transcription time:	3.85 s


{'audio_file': './data/input/sub-DC22212_ses-v1_task-wab-audio.wav',
 'torch_backend': 'cuda',
 'torch_dtype': 'torch.float32',
 'audio_duration_s': 52.984489795918364,
 'pipeline_creation_time_s': 0.0,
 'transcription_time_s': 3.8477821350097656}

In [5]:
def process_folder(
    input_folder: str, output_folder: str, overwrite_existing: bool = False
):

    for file in os.listdir(input_folder):
        if not file.endswith(".wav"):
            continue

        participant_id, task_version, task_name = file.split("_")
        outfile = (
            f"{output_folder}/output/{participant_id}_{task_version}_{task_name}.cha"
        )

        for backend in [TorchBackend.CUDA, TorchBackend.CPU]:

            metadata_file = f"{output_folder}/metadata/{backend.value}_{participant_id}_{task_version}_{task_name}.json"

            if (
                os.path.isfile(metadata_file)
                and os.path.isfile(outfile)
                and not overwrite_existing
            ):
                print(
                    f"> Skipping {file} with backend {backend.value} (already exists)"
                )
                continue

            try:
                result = transcribe_audio(
                    f"{input_folder}/{file}",
                    backend,
                    outfile,
                )

                with open(metadata_file, "w") as f:
                    json.dump(result, f, indent=4)
                print(f"> Transcription metadata saved to {metadata_file}")

            except Exception as e:
                print(f"Error processing {file}: {e}")
                continue


process_folder("./data/input", "./data/batchalign", overwrite_existing=False)

Error processing sub-DC22129_ses-v2_task-wab-audio.wav: 'str' object has no attribute '__dict__'
> Skipping sub-DC22129_ses-v2_task-wab-audio.wav with backend cpu (already exists)
Error processing sub-DC22214_ses-v1_task-wab-audio.wav: 'str' object has no attribute '__dict__'
> Skipping sub-DC22214_ses-v1_task-wab-audio.wav with backend cpu (already exists)
Error processing sub-DC22129_ses-v2_task-sandwich-audio.wav: 'str' object has no attribute '__dict__'
> Skipping sub-DC22129_ses-v2_task-sandwich-audio.wav with backend cpu (already exists)
Error processing sub-DC22214_ses-v1_task-sandwich-audio.wav: 'str' object has no attribute '__dict__'
> Skipping sub-DC22214_ses-v1_task-sandwich-audio.wav with backend cpu (already exists)
Error processing sub-DC22124_ses-v1_task-sandwich-audio.wav: 'str' object has no attribute '__dict__'
> Skipping sub-DC22124_ses-v1_task-sandwich-audio.wav with backend cpu (already exists)
Error processing sub-DC22216_ses-v1_task-sandwich-audio.wav: 'str' obj

In [6]:
def aggregate_results(metadata_folder: str) -> pd.DataFrame:
    data = []
    for file in os.listdir(metadata_folder):

        with open(f"{metadata_folder}/{file}", "r") as f:
            d = json.load(f)
        data += [d]

    df = pd.DataFrame(data)
    df["audio_file"] = df["audio_file"].str.lstrip("./data/input/")
    df["real_time_factor"] = 1 / (df["transcription_time_s"] / df["audio_duration_s"])
    del df["torch_dtype"]
    del df["pipeline_creation_time_s"]
    return df.sort_values(["audio_file", "torch_backend"])


df = aggregate_results("./data/batchalign/metadata")
df.to_csv("./data/batchalign/batchalign_results.csv", index=False)
df

,audio_file,torch_backend,audio_duration_s,transcription_time_s,real_time_factor
11,sub-DC22112_ses-v1_task-sandwich-audio.wav,cpu,42.202018,14.665433,2.877652
38,sub-DC22112_ses-v1_task-sandwich-audio.wav,mps,42.202018,8.425200,5.009023
12,sub-DC22124_ses-v1_task-sandwich-audio.wav,cpu,50.005442,14.006974,3.570039
1,sub-DC22124_ses-v1_task-sandwich-audio.wav,mps,50.005442,7.000945,7.142670
18,sub-DC22124_ses-v1_task-wab-audio.wav,cpu,35.170068,11.052836,3.181995
8,sub-DC22124_ses-v1_task-wab-audio.wav,mps,35.170068,4.758924,7.390341
24,sub-DC22127_ses-v2_task-sandwich-audio.wav,cpu,30.629932,24.506065,1.249892
28,sub-DC22127_ses-v2_task-sandwich-audio.wav,mps,30.629932,12.607041,2.429589
5,sub-DC22127_ses-v2_task-wab-audio.wav,cpu,34.829025,45.667033,0.762673
35,sub-DC22127_ses-v2_task-wab-audio.wav,mps,34.829025,14.296769,2.436147
